# Visual Modality: Training & Evaluation

Sequence model (bi-LSTM / GRU) on the cached `(T, D)` visual features from the EDA pipeline. Real data only. All logic lives in `src/training/visual/`.

| Stage | Class | Module |
|---|---|---|
| Load cached features | `CachedFeatureLoader`, `FeatureSelector`, `SpeakerParser` | `dataset.py` |
| Speaker-disjoint split | `SpeakerSplitter` | `splits.py` |
| Normalise + torch dataset | `Standardizer`, `VisualSequenceDataset` | `dataset.py` |
| Model | `RNNClassifier` (`rnn_type="lstm"\|"gru"`) | `model.py` |
| Training | `Trainer` | `trainer.py` |
| Evaluation | `Evaluator` | `evaluator.py` |
| One full run / seed sweep | `Experiment` | `experiment.py` |

Switch LSTM <-> GRU, pooling, or feature subsets (ablations) only in the config cell.

In [ ]:
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"

import pandas as pd
from IPython.display import display

from src.common.constants import OPENFACE_BIN
from src.extractors.visual.visual_feature_extractor import VisualFeatureExtractor
from notebooks.eda.visual import (
    EDAConfig, HFDatasetLoader, VideoCatalog, OpenFaceTableStore, SceneDetector,
    QualityAuditor, QualityThresholds, FeatureStore,
)
from src.training.visual import (
    TrainConfig, CachedFeatureLoader, FeatureSelector, SpeakerParser, SpeakerSplitter,
    Experiment, TrainingPlots, EvalPlots, Evaluator,
)

## Config

In [ ]:
eda_cfg = EDAConfig(data_dir="../data", openface_bin=OPENFACE_BIN, device="cuda")   # same settings as the EDA notebook

cfg = TrainConfig(
    rnn_type="lstm",            # "lstm" | "gru"
    hidden_dim=128, num_layers=2, bidirectional=True, pooling="attention",
    use_quality_gate=False,     # True = only keep videos whose scenes all pass the Stage-3 gate
    include_prefixes=None,      # e.g. ("au_", "gaze_")
    exclude_prefixes=None,      # e.g. ("emo_",) to ablate emotion probabilities
    seed=42,
)
os.makedirs(cfg.output_dir, exist_ok=True)
print(cfg)

## Data: cached OpenFace features
Reuses the EDA caches (`openface_cache`, `feature_cache`); only videos missing from them are computed.

In [ ]:
loader = HFDatasetLoader(eda_cfg.hf_repo_id, eda_cfg.data_dir, folders=eda_cfg.hf_folders)
loader.download()
catalog = VideoCatalog(eda_cfg.data_dir)
meta_df = catalog.build(loader.local_video_paths())

extractor = VisualFeatureExtractor(openface_bin_path=eda_cfg.openface_bin, device=eda_cfg.device)
tables = OpenFaceTableStore(extractor, eda_cfg.openface_cache_dir, workers=eda_cfg.openface_workers)
tables.ensure(meta_df)
features = FeatureStore(extractor, tables, eda_cfg.feature_cache_dir, target_t=eda_cfg.target_t)

In [ ]:
quality_df = None
if cfg.use_quality_gate:
    scenes, _ = SceneDetector(
        eda_cfg.scene_cache_dir, method=eda_cfg.scene_method, threshold=eda_cfg.scene_threshold,
        min_scene_len=eda_cfg.scene_min_len, frame_skip=eda_cfg.scene_frame_skip,
        workers=eda_cfg.scene_workers, jump_mm=eda_cfg.pose_jump_mm).ensure(meta_df, tables)
    quality_df, _ = QualityAuditor(QualityThresholds(
        eda_cfg.conf_thresh, eda_cfg.jitter_thresh, eda_cfg.max_dropout_pct)).audit(meta_df, tables, scenes)

feature_loader = CachedFeatureLoader(features, SpeakerParser())   # pass SpeakerParser(fn) if the truth/ layout differs
data = feature_loader.load(meta_df, quality_df, FeatureSelector(cfg.include_prefixes, cfg.exclude_prefixes))
display(data.describe())
print(f"{data.input_dim} features:", data.feature_names)

## Split preview (speaker-disjoint, label-stratified)

In [ ]:
splits = SpeakerSplitter(cfg.val_frac, cfg.test_frac, cfg.seed).split(data)
display(SpeakerSplitter.summary(data, splits))

## Train

In [ ]:
exp = Experiment(cfg)
result = exp.run(data)
TrainingPlots.curves(result.history)

## Evaluate
Train / val / test, with per-video predictions saved to CSV.

In [ ]:
display(result.metrics)
for split in ["val", "test"]:
    display(Evaluator.confusion(result.predictions, split))
EvalPlots.roc(result.predictions)

result.predictions.to_csv(f"{cfg.output_dir}/predictions_{cfg.rnn_type}_seed{result.seed}.csv", index=False)
result.metrics.to_csv(f"{cfg.output_dir}/metrics_{cfg.rnn_type}_seed{result.seed}.csv")

In [ ]:
display(Evaluator.per_speaker(result.predictions, "test"))
display(result.predictions.query("split == 'test' and not correct").head(20))

## Robustness: repeat over seeds
Each seed draws a new speaker split, so this measures split variance as well as init variance. Single-split test accuracy on a small dataset is noisy.

In [ ]:
sweep = exp.run_seeds(data, seeds=[0, 1, 2, 3, 4])
display(sweep.groupby("split")[["accuracy", "balanced_acc", "f1_macro", "auc"]].agg(["mean", "std"]).round(3))
EvalPlots.seed_sweep(sweep, "accuracy")
sweep.to_csv(f"{cfg.output_dir}/seed_sweep_{cfg.rnn_type}.csv")